In [21]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [32]:
# datasets and dataloarders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# transforms.compose → combines multiple preprocessing steps into one pipeline
# Data will pass through each step in order (top → bottom)
transform = transforms.Compose([

    # ToTensor → converts image (PIL/numpy) to PyTorch tensor
    # Also scales pixel values from [0,255] → [0,1]
    transforms.ToTensor(),

    # Normalize → standardizes data using mean & std
    # Formula: (value - mean) / std
    # (0.5,0.5,0.5) → for 3 channels (RGB)
    # Makes training faster & more stable
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trainset = CIFAR10(root="./data",train=True,download=True,transform=transform) 
testset = CIFAR10(root="./data",train=False,download=True,transform=transform)

In [33]:
trainloader = DataLoader(trainset,batch_size=64,shuffle=True)
testloader = DataLoader(testset,batch_size=64)

### Build the CNN

In [38]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3,32,kernel_size = 3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), #kernel size =2,stride =2

            nn.Conv2d(32,64,kernel_size = 3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), #kernel size =2,stride =2

            nn.Conv2d(64,128,kernel_size = 3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), #kernel size =2,stride =2
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),

            nn.Linear(256,10)
        )

    def forward(self,x):
        x = self.conv_layers(x)
        x = x.view(x.size(0),-1)# flatten
        x = self.fc_layers(x)

        return x

In [39]:
model = CNN()

In [40]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

### Training the CNN


In [41]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()
        
        output = model.forward(images) # FP
        loss = criterion(output, labels) # loss fnx
        loss.backward() # BP
        optimizer.step() # update params

        epoch_training_loss += loss.item()

    print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss/len(trainloader)}")

epoch=1/10 & loss=1.4274890668251936
epoch=2/10 & loss=1.0010005957482722
epoch=3/10 & loss=0.8170861846879315
epoch=4/10 & loss=0.6851334245613468
epoch=5/10 & loss=0.59110775399391
epoch=6/10 & loss=0.5030714298605614
epoch=7/10 & loss=0.4232361036760118
epoch=8/10 & loss=0.35452047152363736
epoch=9/10 & loss=0.29382311894803703
epoch=10/10 & loss=0.23581556428481093


In [42]:
# Evaludate our CNN

correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _, predicted  = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"accuracy = {correct_labels / total_labels * 100}")

accuracy = 75.07000000000001
